# Collaby Qwen SFT + QLoRA

Этот notebook запускается в Google Colab и обучает LoRA-адаптер для `Qwen2.5-1.5B-Instruct` на объединённом датасете `mentor` + `assistant`.
Финальный deploy-ready артефакт для текущего runtime — только GGUF LoRA-файл, совместимый с `llama.cpp`.


In [ ]:
!pip install -qU transformers accelerate bitsandbytes peft trl datasets sentencepiece safetensors gguf huggingface_hub


In [ ]:
from google.colab import files
import json
import zipfile
from pathlib import Path

uploaded = files.upload()
upload_name = next(iter(uploaded.keys()))
bundle_root = Path('/content/collaby-colab-bundle')
bundle_root.mkdir(parents=True, exist_ok=True)

if upload_name.endswith('.zip'):
    with zipfile.ZipFile(upload_name) as archive:
        archive.extractall(bundle_root)
    bundle_manifest_path = next(bundle_root.glob('*/bundle-manifest.json'))
    bundle_dir = bundle_manifest_path.parent
    bundle_manifest = json.loads(bundle_manifest_path.read_text(encoding='utf-8'))
    DATASET_PATH = str(bundle_dir / 'dataset' / 'collaby-qwen-sft.jsonl')
    EXPORT_SCRIPT_PATH = str(bundle_dir / 'scripts' / 'export_adapter_manifest.py')
    ADAPTER_TEMPLATE_PATH = str(bundle_dir / 'runtime' / 'adapter-manifest.template.json')
else:
    bundle_manifest = None
    DATASET_PATH = upload_name
    EXPORT_SCRIPT_PATH = ''
    ADAPTER_TEMPLATE_PATH = ''

{
    'dataset': DATASET_PATH,
    'export_script': EXPORT_SCRIPT_PATH,
    'adapter_template': ADAPTER_TEMPLATE_PATH,
}

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
dataset = load_dataset('json', data_files=DATASET_PATH, split='train')

def format_example(example):
    chunks = []
    for message in example['messages']:
        chunks.append(f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>")
    return {'text': '\n'.join(chunks)}

dataset = dataset.map(format_example)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype='float16',
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

training_args = SFTConfig(
    output_dir='collaby-qwen-lora',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy='epoch',
    dataset_text_field='text',
    bf16=False,
    fp16=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
)

trainer.train()
trainer.model.save_pretrained('collaby-qwen-lora/final-adapter')
tokenizer.save_pretrained('collaby-qwen-lora/final-adapter')

In [ ]:
import subprocess
import sys
from pathlib import Path

LLAMA_CPP_DIR = Path('/content/llama.cpp')
if not LLAMA_CPP_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_CPP_DIR)],
        check=True,
    )

GGUF_ADAPTER_PATH = Path('collaby-qwen-lora/final-adapter-f16.gguf')
subprocess.run(
    [
        sys.executable,
        str(LLAMA_CPP_DIR / 'convert_lora_to_gguf.py'),
        '--base-model-id',
        MODEL_ID,
        '--outfile',
        str(GGUF_ADAPTER_PATH),
        '--outtype',
        'f16',
        'collaby-qwen-lora/final-adapter',
    ],
    check=True,
)
GGUF_ADAPTER_PATH


Финальный deploy-ready артефакт для `api-ai` — `collaby-qwen-lora/final-adapter-f16.gguf`.

Если notebook запускался из repo-managed bundle, скачайте именно этот `.gguf`-файл, затем на локальной машине выполните `export_adapter_manifest.py` из того же bundle и обновите `services/AgentService/adapter-manifest.json`.

Raw каталог `collaby-qwen-lora/final-adapter/` и его `.zip` можно хранить только как вспомогательный training-артефакт, но не как runtime-адаптер для `llama.cpp`.
